In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [ ]:
BERTOPIC_DATA_DIR = Path('/mimer/NOBACKUP/groups/naiss2024-6-297/cache/bertopic_data/bertopic_inputs')
CACHE_DIR = Path('/mimer/NOBACKUP/groups/naiss2024-6-297/cache/bertopic_bootstrapped/full_training_10')
TRAIN_MODEL_DIR = CACHE_DIR / 'models'
INFERENCE_DIR = CACHE_DIR / 'inference'
NR_TOPICS_LIST = list(range(5, 85, 5))  # [5, 10, 15, ..., 80]
N_BOOTSTRAPS = 10

metadata = pd.read_csv(BERTOPIC_DATA_DIR / 'all_stories_metadata.csv')
with open(BERTOPIC_DATA_DIR / 'all_stories_texts.json', 'r') as f:
    stories = json.load(f)

print(f'Stories: {len(stories)}, Metadata rows: {len(metadata)}')
print(f'Columns: {list(metadata.columns)}')
metadata.head()

In [ ]:
# build sentence-level mapping
def build_sentence_mapping(stories, metadata):
    rows = []
    current_idx = 0
    for doc_id, doc in enumerate(stories):
        sentences = [s.strip() for s in doc.split('[SENT]') if s.strip()]
        meta = metadata[metadata['story_index'] == doc_id].iloc[0]
        for pos, sent in enumerate(sentences):
            rows.append({
                'sentence_idx': current_idx, 'sentence_pos': pos,
                'sentence_text': sent,
                'n_sentences': len(sentences), 'story_index': doc_id,
                'story_id': meta['doc_key'],
                'model_type': meta['model_type'],
                'prompt_type': meta['prompt_type'],
                'seed': meta['seed'],
            })
            current_idx += 1
    return pd.DataFrame(rows)

def filter_by_seed(df):
    """Filter to canonical seed selections:
    - original prompt: seed=42 only
    - large prompt, AI models: seed=42 only
    - large prompt, human: all seeds (42, 43, 44)
    """
    original = df[(df['prompt_type'] == 'original') & (df['seed'] == 42)]
    large_ai = df[(df['prompt_type'] == 'large') & (df['model_type'] != 'human') & (df['seed'] == 42)]
    large_human = df[(df['prompt_type'] == 'large') & (df['model_type'] == 'human')]
    return pd.concat([original, large_ai, large_human], ignore_index=True)

sent_map = build_sentence_mapping(stories, metadata)
print(f'Total sentences: {len(sent_map)}')
sent_map_filtered = filter_by_seed(sent_map)
print(f'Filtered sentences (seed rules): {len(sent_map_filtered)}')
sent_map.head()

In [ ]:
available_train = []
available_infer = []

for b in range(N_BOOTSTRAPS):
    for nt in NR_TOPICS_LIST:
        model_dir = TRAIN_MODEL_DIR / f'bootstrap_{b:02d}' / f'topics_{nt}'
        infer_dir = INFERENCE_DIR / f'bootstrap_{b:02d}' / f'topics_{nt}'
        if model_dir.exists() and (model_dir / 'topic_info.csv').exists():
            available_train.append((b, nt))
        if infer_dir.exists() and (infer_dir / 'sentence_topics.npy').exists():
            available_infer.append((b, nt))

print(f'Trained models available: {len(available_train)} / {N_BOOTSTRAPS * len(NR_TOPICS_LIST)}')
print(f'Inference results available: {len(available_infer)} / {N_BOOTSTRAPS * len(NR_TOPICS_LIST)}')

# completion grid
train_grid = pd.DataFrame(0, index=range(N_BOOTSTRAPS), columns=NR_TOPICS_LIST)
infer_grid = pd.DataFrame(0, index=range(N_BOOTSTRAPS), columns=NR_TOPICS_LIST)
for b, nt in available_train:
    train_grid.loc[b, nt] = 1
for b, nt in available_infer:
    infer_grid.loc[b, nt] = 1

print('\nTraining completion (rows=bootstrap, cols=nr_topics):')
display(train_grid)
print('\nInference completion:')
display(infer_grid)

In [ ]:
# pick a config to inspect
BOOTSTRAP = 0
NR_TOPICS = 80

topic_info_path = TRAIN_MODEL_DIR / f'bootstrap_{BOOTSTRAP:02d}' / f'topics_{NR_TOPICS}' / 'topic_info.csv'

if not topic_info_path.exists():
    print(f'Not found: {topic_info_path}')
else:
    topic_info = pd.read_csv(topic_info_path)
    print(f'bootstrap={BOOTSTRAP}, nr_topics={NR_TOPICS}')
    print(f'Topics (incl outlier -1): {len(topic_info)}')
    print(f'Columns: {list(topic_info.columns)}')
    print()

    # Parse top words from Name column
    def extract_top_words(name, max_words=20):
        if pd.isna(name):
            return ''
        parts = str(name).split('_')
        if len(parts) > 1:
            return ', '.join(parts[1:1+max_words])
        return str(name)

    topic_info['TopWords'] = topic_info['Name'].apply(extract_top_words)

    display_cols = ['Topic', 'Count']
    if 'Name' in topic_info.columns:
        display_cols.append('Name')
    display_cols.append('TopWords')

    with pd.option_context('display.max_colwidth', None):
        display(topic_info[display_cols])

In [ ]:
# topic size distribution
if topic_info_path.exists():
    main_topics = topic_info[topic_info['Topic'] != -1]
    outlier_row = topic_info[topic_info['Topic'] == -1]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(range(len(main_topics)), main_topics['Count'].values, color='steelblue', alpha=0.8)
    ax.set_xlabel('Topic (sorted by size)')
    ax.set_ylabel('Sentences in topic')
    ax.set_title(f'Topic sizes — bootstrap={BOOTSTRAP}, nr_topics={NR_TOPICS}')

    if not outlier_row.empty:
        outlier_count = int(outlier_row['Count'].iloc[0])
        total = int(topic_info['Count'].sum())
        ax.axhline(y=outlier_count, color='red', linestyle='--', alpha=0.6,
                   label=f'Outlier topic -1: {outlier_count} ({outlier_count/total:.1%})')
        ax.legend()

    fig.tight_layout()
    plt.show()

In [ ]:
STORY_ID = 10408          # change to any story_id
MODEL_TYPE = 'human'   # model_type from metadata
PROMPT_TYPE = 'large'
SEED = 44
BOOTSTRAP = 0
NR_TOPICS = 80

# Load inference sentence topics
infer_path = INFERENCE_DIR / f'bootstrap_{BOOTSTRAP:02d}' / f'topics_{NR_TOPICS}' / 'sentence_topics.npy'
topic_info_path_inspect = TRAIN_MODEL_DIR / f'bootstrap_{BOOTSTRAP:02d}' / f'topics_{NR_TOPICS}' / 'topic_info.csv'

if not infer_path.exists():
    print(f'Inference not found: {infer_path}')
else:
    sentence_topics = np.load(infer_path)

    # Load topic info for names/top words
    topic_name_map = {}
    topic_words_map = {}
    if topic_info_path_inspect.exists():
        ti = pd.read_csv(topic_info_path_inspect)
        for _, row in ti.iterrows():
            tid = row['Topic']
            name = str(row.get('Name', ''))
            parts = name.split('_')
            top_words = ', '.join(parts[1:21]) if len(parts) > 1 else name
            topic_name_map[tid] = name
            topic_words_map[tid] = top_words

    # Find the story's sentences
    story_sents = sent_map[
        (sent_map['story_id'].astype(str) == str(STORY_ID))
        & (sent_map['model_type'] == MODEL_TYPE)
        & (sent_map['prompt_type'] == PROMPT_TYPE)
        & (sent_map['seed'] == SEED)
    ].sort_values('sentence_pos').copy()

    if story_sents.empty:
        print(f'No sentences found for story_id={STORY_ID}, model={MODEL_TYPE}, prompt={PROMPT_TYPE}, seed={SEED}')
    else:
        story_sents['topic'] = sentence_topics[story_sents['sentence_idx'].values]
        story_sents['topic_name'] = story_sents['topic'].map(topic_name_map).fillna('')
        story_sents['top_words'] = story_sents['topic'].map(topic_words_map).fillna('')

        print(f'Story {STORY_ID} | {MODEL_TYPE} / {PROMPT_TYPE} / seed={SEED}')
        print(f'bootstrap={BOOTSTRAP}, nr_topics={NR_TOPICS}')
        print(f'Sentences: {len(story_sents)}, Unique topics assigned: {story_sents["topic"].nunique()}')
        print()

        # Show each sentence with its topic, name, and top words
        with pd.option_context('display.max_colwidth', None, 'display.max_columns', None, 'display.width', None, 'display.max_rows', None):
            display(story_sents[['sentence_pos', 'topic', 'topic_name', 'top_words', 'sentence_text']])

In [ ]:
if infer_path.exists() and not story_sents.empty:
    topics_seq = story_sents['topic'].values
    positions = story_sents['sentence_pos'].values
    unique_topics = sorted(set(topics_seq))

    cmap = plt.cm.get_cmap('tab20', max(len(unique_topics), 1))
    topic_to_color = {t: cmap(i) for i, t in enumerate(unique_topics)}
    colors = [topic_to_color[t] for t in topics_seq]

    fig, ax = plt.subplots(figsize=(max(12, len(topics_seq) * 0.5), 2))
    ax.barh(0, width=1, left=positions, color=colors, edgecolor='white', linewidth=0.5,
            height=0.8)

    for pos, t in zip(positions, topics_seq):
        ax.text(pos + 0.5, 0, str(t), ha='center', va='center', fontsize=7, fontweight='bold')

    ax.set_xlim(-0.5, len(topics_seq) + 0.5)
    ax.set_xlabel('Sentence position')
    ax.set_yticks([])
    ax.set_title(f'Topic sequence — story {STORY_ID} ({MODEL_TYPE}/{PROMPT_TYPE}) | nr_topics={NR_TOPICS}')

    from matplotlib.patches import Patch
    legend_patches = [Patch(facecolor=topic_to_color[t], label=f'Topic {t}') for t in unique_topics]
    ax.legend(handles=legend_patches, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)

    fig.tight_layout()
    plt.show()

In [ ]:
STORY_ID_CMP = 10408
MODEL_TYPE_CMP = 'human'
PROMPT_TYPE_CMP = 'large'
SEED_CMP = 42
BOOTSTRAP_CMP = 0

story_sents_base = sent_map[
    (sent_map['story_id'].astype(str) == str(STORY_ID_CMP))
    & (sent_map['model_type'] == MODEL_TYPE_CMP)
    & (sent_map['prompt_type'] == PROMPT_TYPE_CMP)
    & (sent_map['seed'] == SEED_CMP)
].sort_values('sentence_pos').copy()

if story_sents_base.empty:
    print('Story not found.')
else:
    n_sents = len(story_sents_base)
    sent_indices = story_sents_base['sentence_idx'].values

    # Collect topic assignments per nr_topics
    rows_nt = []
    for nt in NR_TOPICS_LIST:
        ip = INFERENCE_DIR / f'bootstrap_{BOOTSTRAP_CMP:02d}' / f'topics_{nt}' / 'sentence_topics.npy'
        if not ip.exists():
            continue
        topics = np.load(ip)
        for pos, sidx in enumerate(sent_indices):
            rows_nt.append({'nr_topics': nt, 'sentence_pos': pos, 'topic': int(topics[sidx])})

    df_nt = pd.DataFrame(rows_nt)

    if df_nt.empty:
        print('No inference results found.')
    else:
        # Pivot: rows = nr_topics, columns = sentence_pos
        pivot = df_nt.pivot(index='nr_topics', columns='sentence_pos', values='topic')

        # Count topic switches per nr_topics
        switches = []
        for nt in pivot.index:
            seq = pivot.loc[nt].values
            n_changes = sum(1 for i in range(len(seq) - 1) if seq[i] != seq[i+1])
            switches.append({'nr_topics': nt, 'n_switches': n_changes,
                             'switch_rate': np.tanh(n_changes / (len(seq) - 1)) if len(seq) > 1 else 0})
        switches_df = pd.DataFrame(switches)

        print(f'Story {STORY_ID_CMP} ({MODEL_TYPE_CMP}/{PROMPT_TYPE_CMP}), {n_sents} sentences')
        print(f'bootstrap={BOOTSTRAP_CMP}')
        print()
        print('Topic switches by nr_topics:')
        display(switches_df)
        print()
        print('Topic assignments (rows=nr_topics, cols=sentence_pos):')
        with pd.option_context('display.max_columns', None):
            display(pivot)

In [ ]:
if not df_nt.empty:
    fig, ax = plt.subplots(figsize=(max(12, n_sents * 0.6), len(pivot) * 0.5 + 1))

    all_topic_ids = sorted(df_nt['topic'].unique())
    n_colors = max(len(all_topic_ids), 1)
    cmap = plt.cm.get_cmap('tab20', n_colors)
    topic_to_idx = {t: i for i, t in enumerate(all_topic_ids)}

    matrix = pivot.values.copy().astype(float)
    matrix_mapped = np.vectorize(lambda t: topic_to_idx.get(int(t), -1))(matrix)

    im = ax.imshow(matrix_mapped, aspect='auto', cmap=cmap, interpolation='nearest')

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, str(int(matrix[i, j])), ha='center', va='center', fontsize=6)

    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels(pivot.index)
    ax.set_ylabel('nr_topics')
    ax.set_xlabel('Sentence position')
    ax.set_title(f'Topic assignments — story {STORY_ID_CMP} ({MODEL_TYPE_CMP}/{PROMPT_TYPE_CMP})')

    fig.tight_layout()
    plt.show()

In [ ]:
AGG_BOOTSTRAP = 0
AGG_NR_TOPICS = 30

agg_infer_path = INFERENCE_DIR / f'bootstrap_{AGG_BOOTSTRAP:02d}' / f'topics_{AGG_NR_TOPICS}' / 'sentence_topics.npy'

if not agg_infer_path.exists():
    print(f'Not found: {agg_infer_path}')
else:
    all_topics = np.load(agg_infer_path)

    # Use seed-filtered sentences only
    filtered_indices = sent_map_filtered['sentence_idx'].values
    filtered_topics = all_topics[filtered_indices]
    topic_counts = pd.Series(filtered_topics).value_counts().sort_index()

    print(f'bootstrap={AGG_BOOTSTRAP}, nr_topics={AGG_NR_TOPICS} (seed-filtered)')
    print(f'Total sentences (filtered): {len(filtered_topics)}')
    print(f'Unique topics assigned: {len(topic_counts)}')
    print(f'Outlier (topic -1): {(filtered_topics == -1).sum()} ({(filtered_topics == -1).mean():.1%})')

    fig, ax = plt.subplots(figsize=(12, 4))
    non_outlier = topic_counts[topic_counts.index != -1]
    ax.bar(non_outlier.index.astype(str), non_outlier.values, color='steelblue', alpha=0.8)
    ax.set_xlabel('Topic ID')
    ax.set_ylabel('Sentence count')
    ax.set_title(f'Topic frequency (inference, seed-filtered) — bootstrap={AGG_BOOTSTRAP}, nr_topics={AGG_NR_TOPICS}')
    plt.xticks(rotation=90, fontsize=7)
    fig.tight_layout()
    plt.show()

In [ ]:
# topic distribution by condition (seed-filtered)
if agg_infer_path.exists():
    df_with_topics = sent_map_filtered.copy()
    df_with_topics['topic'] = all_topics[df_with_topics['sentence_idx'].values]

    # Topic counts by model_type x prompt_type
    condition_topics = (
        df_with_topics.groupby(['model_type', 'prompt_type', 'topic'])
        .size()
        .reset_index(name='count')
    )

    # Outlier rate by condition
    outlier_rates = (
        df_with_topics.groupby(['model_type', 'prompt_type'])
        .apply(lambda g: (g['topic'] == -1).mean())
        .reset_index(name='outlier_rate')
        .sort_values('outlier_rate', ascending=False)
    )

    print(f'Outlier rate by condition (bootstrap={AGG_BOOTSTRAP}, nr_topics={AGG_NR_TOPICS}, seed-filtered):')
    display(outlier_rates)

    # Unique topics per condition
    unique_per_cond = (
        df_with_topics[df_with_topics['topic'] != -1]
        .groupby(['model_type', 'prompt_type'])['topic']
        .nunique()
        .reset_index(name='n_unique_topics')
        .sort_values('n_unique_topics', ascending=False)
    )
    print(f'\nUnique topics per condition:')
    display(unique_per_cond)

In [ ]:
CMP_STORY_ID = 10408
CMP_PROMPT = 'large'
CMP_BOOTSTRAP = 0
CMP_NR_TOPICS = 30

cmp_infer_path = INFERENCE_DIR / f'bootstrap_{CMP_BOOTSTRAP:02d}' / f'topics_{CMP_NR_TOPICS}' / 'sentence_topics.npy'

if not cmp_infer_path.exists():
    print(f'Not found: {cmp_infer_path}')
else:
    all_topics_cmp = np.load(cmp_infer_path)

    # Get all models that have this story_id + prompt (seed-filtered)
    story_all_models = sent_map_filtered[
        (sent_map_filtered['story_id'].astype(str) == str(CMP_STORY_ID))
        & (sent_map_filtered['prompt_type'] == CMP_PROMPT)
    ].copy()
    story_all_models['topic'] = all_topics_cmp[story_all_models['sentence_idx'].values]

    models_available = sorted(story_all_models['model_type'].unique())
    print(f'Story {CMP_STORY_ID}, prompt={CMP_PROMPT}, nr_topics={CMP_NR_TOPICS} (seed-filtered)')
    print(f'Models available: {models_available}')
    print()

    for model in models_available:
        sub = story_all_models[story_all_models['model_type'] == model].sort_values('sentence_pos')
        seeds = sorted(sub['seed'].unique())
        # For human-large: show per-seed then average
        if model == 'human' and len(seeds) > 1:
            all_srs = []
            for seed in seeds:
                sub_seed = sub[sub['seed'] == seed].sort_values('sentence_pos')
                topics_seq = sub_seed['topic'].values
                n_sw = sum(1 for i in range(len(topics_seq) - 1) if topics_seq[i] != topics_seq[i+1])
                sr = np.tanh(n_sw / (len(topics_seq) - 1)) if len(topics_seq) > 1 else 0
                all_srs.append(sr)
                print(f'{model:>20s} (seed={seed}): {len(sub_seed)} sents, {len(set(topics_seq))} unique topics, '
                      f'{n_sw} switches, switch_rate={sr:.3f}')
                print(f'{" " * 22}topics: {list(topics_seq)}')
            print(f'{model:>20s}   avg switch_rate={np.mean(all_srs):.3f}')
        else:
            topics_seq = sub['topic'].values
            n_sw = sum(1 for i in range(len(topics_seq) - 1) if topics_seq[i] != topics_seq[i+1])
            sr = np.tanh(n_sw / (len(topics_seq) - 1)) if len(topics_seq) > 1 else 0
            print(f'{model:>20s}: {len(sub)} sents, {len(set(topics_seq))} unique topics, '
                  f'{n_sw} switches, switch_rate={sr:.3f}')
            print(f'{" " * 22}topics: {list(topics_seq)}')
        print()